In [2]:
import pandas as pd
import numpy as np
import os
import sys
from notebook.services.config import ConfigManager
cm = ConfigManager().update('notebook', {'limit_output': 1000})
pd.options.display.max_rows=999
pd.options.display.max_columns=999
%matplotlib widget
import matplotlib.pyplot as plt
from edw_functions import *
sys.path.append('/home/wolfgang/repos/sleep_research_io/')
from sleep_research_functions import *
from sleep_analysis_functions import *
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def read_in_cohort(filepath_cohort):
    
    cohort = pd.read_csv(filepath_cohort)
    cohort.dropna(subset=['MRN:'], inplace=True)
    cohort['study_id'] = cohort['Study ID:'].apply(lambda x: str(x).zfill(3))
    cohort['mrn'] = cohort['MRN:'].apply(lambda x: int(x))

    return cohort

In [4]:
def load_sofa_data(filepath_sofa):
    
    sofa_data = pd.read_csv(filepath_sofa)
    sofa_data.columns = ['sofa_respiratory','sofa_coagulatory', 'sofa_liver', 'sofa_cvs', 'sofa_cns', 'sofa_renal', 'sofa_score', 'sofa_delta_2h', 'sofa_delta_4h', 'sofa_delta_6h', 'datetime']
    sofa_data['datetime'] = pd.to_datetime(sofa_data['datetime'], infer_datetime_format=1)
    sofa_data.set_index('datetime', inplace=True)
    
    return sofa_data
    

def merge_with_data_sofa(data, study_id, dir_sofa):
    '''
    inputs:
    data: data (dataframe) to be merged with.
    study_id: study_id the data belongs to SOFA of this study_id will be merged.
    dir_sofa: directory containing all SOFA files, as output from matlab sofa computation script.
    outputs:
    data: dataframe with SOFA
    [Note: SOFA is converted to integer, with -99 for NaNs.]
    '''
    
    filepath_sofa = os.path.join(dir_sofa, 'SOFA_ICU'+study_id+'.csv')

    sofa_data = load_sofa_data(filepath_sofa)
    sofa_data = sofa_data.loc[str(data.index[0].replace(microsecond=0)) : str(data.index[-1].replace(microsecond=0)),:]
    sofa_cols = list(sofa_data.columns)
    
    data = data.join(sofa_data)

    for col_tmp in sofa_cols:
        data.loc[pd.isna(data[col_tmp]), col_tmp] = -99
        data.loc[:, col_tmp] = data[col_tmp].round().astype(np.int32)

    return data


In [5]:
def merge_with_data_labs(data, mrn, filepath_labs):
    '''
    inputs:
    data: data (dataframe) to be merged with.
    mrn: mrn the data belongs to, labs of this mrn will be merged.
    filepath_labs: filepath to raw EDW labs.
    outputs:
    data: dataframe with labs. 
    [Note: labs are converted to integer, with -1 for NaNs.]
    '''

    labs = get_labs_single_mrn(filepath_labs, mrn)
    
    numeric = ['hco3_arterial', 'hco3_venous', 'pco2_arterial',
           'pco2_venous', 'ph_arterial', 'ph_venous', 'po2_arterial',
           'po2_venous']
    for col in numeric:
        labs.loc[(labs[col]>9000).values, col] = np.nan
        
    labs = labs.loc[str(data.index[0]) : str(data.index[-1]),:]
    data = data.join(labs)

    int_labs = ['hco3_arterial', 'hco3_venous', 'pco2_arterial', 'pco2_venous', 'po2_arterial', 'po2_venous']
    for lab_tmp in int_labs:
        data.loc[pd.isna(data[lab_tmp]), lab_tmp] = -1
        data.loc[:, lab_tmp] = data[lab_tmp].round().astype(np.int32)
        
    return data

In [6]:
def merge_with_data_meds(data, study_id, dir_meds):
    '''
    inputs:
    data: data (dataframe) to be merged with.
    study id: study_id to be merged.
    dir_meds: directory with the timeseries meds .h5 files, as ouput from meds pipeline.
    outputs:
    data: dataframe with meds. 
    '''
        
    meds_to_add = ['opioids_sum', 'benzos_sum', 'antipsychotics_sum','dexmedetomidine_or_placebo_(2017p000090)']

    for med_to_add in meds_to_add: # remove if already contained. update with new version potentially.
        if med_to_add in data.columns:
            data.drop(med_to_add, axis=1, inplace=True)
    
    med_path = os.path.join(dir_meds, 'icu_' + study_id + '_meds.h5')
    meds_data, hdr_meds = load_sleep_data(med_path, load_all_signals=0, idx_to_datetime=1,
                                          signals_to_load=meds_to_add)
    fs_meds = hdr_meds['fs']
    
    meds_to_add = [x for x in meds_to_add if x in meds_data.columns]

    meds_data = meds_data[meds_to_add]
    meds_data.rename({'dexmedetomidine_or_placebo_(2017p000090)': 'dex_studydrug'}, axis=1, inplace=True)
    
    data = data.join(meds_data, how='left')
    
    return data

In [7]:
def load_cams_scores(filepath_cam):
    
    cams_scores = pd.read_csv(filepath_cam) 
    cams_scores = cams_scores[cams_scores['Study ID:'] != '98a']
    cams_scores['Study ID:'] = cams_scores['Study ID:'].apply(lambda x: x.zfill(3))
    cams_scores = cams_scores.loc[~pd.isna(cams_scores['Total points of CAM-S'])]
    cams_scores.rename({'Evaluation Date and Time': 'datetime'}, axis=1, inplace=True)
    cams_scores.rename({'Total points of CAM-S': 'CAM_S'}, axis=1, inplace=True)
    
    return cams_scores

def merge_with_data_cam(data, study_id, filepath_cam):
    '''
    inputs:
    data: data (dataframe) to be merged with.
    study id: study_id to be merged.
    filepath_cam: filepath to CAM S scores.
    outputs:
    data: dataframe with cam s. 
    [Note: CAM S are converted to integer, with -1 for NaNs.]
    '''
    
    cams_scores = load_cams_scores(filepath_cam)
    
    cam_scores_studyid = cams_scores[cams_scores['Study ID:'] == study_id]
    cam_scores_studyid = cam_scores_studyid.set_index('datetime')
    cam_scores_studyid = cam_scores_studyid[['CAM_S']]
    cam_scores_studyid.rename({'CAM_S': 'cam_s'}, axis=1, inplace=True)
    
    data = data.join(cam_scores_studyid)
    
    data.loc[pd.isna(data['cam_s']), 'cam_s'] = -1
    data.loc[:, 'cam_s'] = data['cam_s'].round().astype(np.int32)
    
    return data

In [8]:
def compute_hypoxia_burden_routine(data):

    apnea_name = 'apnea_pred_ro_a3'  # name of Apnea info column
    hours_sleep = 'stage_pred_comb_breath_activity_1'  # name of sleep stage column, or int/float for manual setting.
    hypoxia_name = 'hypoxic_area' + apnea_name.replace('apnea_pred', '')
    if hypoxia_name in data.columns: 
        data.drop(hypoxia_name, axis=1, inplace=True)

    data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, 
        hypoxia_name=hypoxia_name, hours_sleep = hours_sleep)

    apnea_name = 'apnea_pred_rsr_a3'  # name of Apnea info column
    hours_sleep = 'stage_pred_comb_breath_activity_1'  # name of sleep stage column, or int/float for manual setting.
    hypoxia_name = 'hypoxic_area' + apnea_name.replace('apnea_pred', '')
    if hypoxia_name in data.columns: 
        data.drop(hypoxia_name, axis=1, inplace=True)

    data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, 
        hypoxia_name=hypoxia_name, hours_sleep = hours_sleep)

    apnea_name = 'apnea_pred_va_a3'  # name of Apnea info column
    hours_sleep = 'stage_pred_comb_breath_activity_1'  # name of sleep stage column, or int/float for manual setting.
    hypoxia_name = 'hypoxic_area' + apnea_name.replace('apnea_pred', '')

    data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, 
        hypoxia_name=hypoxia_name, hours_sleep = hours_sleep)
    if hypoxia_name in data.columns: 
        data.drop(hypoxia_name, axis=1, inplace=True)

    data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, 
        hypoxia_name=hypoxia_name, hours_sleep = hours_sleep)
    
    return data


### uniform way to merge specific dataset with overall data:

data = load()

data = merge_with_data_labs()

data = merge_with_data_cam()

data = merge_with_data_sofa()

data = merge_with_data_meds()

def merge_with_data_x(data, filepath_x):
    
    x = load(filepath_x)
    
    x = process_x_for_merge(x)
    
    data = data.join()
    
    return data
    

In [10]:
# directories with data: 
dir_input = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_v2'
dir_sofa = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/SOFA_results_studyid'
dir_meds = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/medications_all_timeseries'
filepath_labs = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/edw_data/edw_icu_sleep_2020_07_07_labs.csv'
filepath_cam = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/ICUSLEEP-CAMSOverallScores.csv'
filepath_cohort = '/media/mad3/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/LIST.csv'

# after step1 where we computed all outputs of models etc., step 2 HRV/CPC, now step3:
dir_output = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step3_nohrv'

overwrite = False
    

In [11]:
if not os.path.exists(dir_output):
    os.makedirs(dir_output)

files = os.listdir(dir_input)
files.sort()

cohort = read_in_cohort(filepath_cohort)

for file in files: # ['icusleep_004.h5']:

    try:
        study_id = file[-6:-3]
        mrn = cohort.loc[cohort.study_id==study_id, 'mrn'].values[0]
        filepath_input = os.path.join(dir_input, file)
        filepath_output = os.path.join(dir_output, file)

        if os.path.exists(filepath_output):
            continue

        print(file)

        data, hdr, fs  = read_in_routine(filepath_input, check_airgo_scaling=False)

        data = merge_with_data_labs(data, mrn, filepath_labs)

        data = merge_with_data_cam(data, study_id, filepath_cam)

        data = merge_with_data_sofa(data, study_id, dir_sofa)

        data = merge_with_data_meds(data, study_id, dir_meds)

        # new computation of hypoxia burden:
        data = compute_hypoxia_burden_routine(data)

        data, hdr = airgo_signal_quality_routine(data, hdr, fs)

        write_to_hdf5_file(data, filepath_output, hdr=hdr, overwrite=overwrite)
        
    except Exception as e:
        print(e)
        continue


icusleep_001.h5


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:12: DtypeWarning: Columns (29,35) have mixed types. Specify dtype option on import or set low_memory=False.
  if sys.path[0] == '':


AirGo V14, open belt threshold deactivated
icusleep_003.h5


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/numpy/lib/nanfunctions.py:1391: RuntimeWarning: All-NaN slice encountered
  overwrite_input, interpolation)


AirGo V14, open belt threshold deactivated
icusleep_004.h5
AirGo V14, open belt threshold deactivated
icusleep_006.h5
AirGo V14, open belt threshold deactivated
icusleep_007.h5
AirGo V14, open belt threshold deactivated
icusleep_011.h5
AirGo V14, open belt threshold deactivated
icusleep_012.h5


...

icusleep_063.h5
icusleep_064.h5


/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:132: RuntimeWarning: Mean of empty slice
  mean_spo2 = np.nanmean(np.array(spo2_collection), axis=0)


icusleep_067.h5
icusleep_068.h5
icusleep_069.h5
icusleep_071.h5
icusleep_072.h5
icusleep_074.h5
icusleep_076.h5
icusleep_077.h5
icusleep_078.h5
icusleep_079.h5
icusleep_080.h5
icusleep_081.h5
icusleep_082.h5
icusleep_083.h5
icusleep_084.h5
icusleep_085.h5
icusleep_086.h5
icusleep_087.h5
icusleep_088.h5
icusleep_089.h5
icusleep_090.h5
icusleep_091.h5
icusleep_092.h5
icusleep_093.h5
icusleep_095.h5
icusleep_097.h5
icusleep_100.h5
icusleep_101.h5
icusleep_102.h5
icusleep_103.h5
icusleep_104.h5
icusleep_105.h5
icusleep_106.h5
icusleep_107.h5
icusleep_108.h5
icusleep_109.h5
icusleep_110.h5
icusleep_111.h5
icusleep_112.h5
icusleep_114.h5
icusleep_115.h5
icusleep_117.h5
icusleep_119.h5
icusleep_120.h5
icusleep_121.h5
icusleep_122.h5
icusleep_123.h5
icusleep_124.h5


/home/wolfgang/repos/ICU-Sleep/code1/sleep_analysis_functions.py:177: RuntimeWarning: divide by zero encountered in double_scalars
  hypoxic_burden = np.sum(areas_robust)/hours_sleep


icusleep_125.h5
icusleep_126.h5
icusleep_128.h5
icusleep_129.h5
icusleep_130.h5
icusleep_131.h5
icusleep_133.h5
icusleep_135.h5
icusleep_137.h5
icusleep_139.h5
icusleep_140.h5
icusleep_142.h5
icusleep_143.h5
icusleep_145.h5
icusleep_149.h5
icusleep_151.h5
icusleep_152.h5
icusleep_153.h5
icusleep_154.h5
icusleep_155.h5
icusleep_156.h5
icusleep_157.h5
icusleep_158.h5
icusleep_160.h5
icusleep_161.h5
icusleep_164.h5
icusleep_165.h5
icusleep_167.h5
icusleep_168.h5
icusleep_169.h5
icusleep_171.h5
icusleep_172.h5
icusleep_173.h5
icusleep_175.h5
icusleep_176.h5
icusleep_177.h5
icusleep_178.h5
icusleep_179.h5
icusleep_180.h5
icusleep_181.h5
icusleep_182.h5
icusleep_183.h5
icusleep_185.h5
icusleep_186.h5
icusleep_187.h5
icusleep_188.h5
icusleep_189.h5


In [ ]:
all_labs =  get_labs(filepath_edw_labs, load_raw=True)


In [27]:
all_labs.columns

Index(['MRN', 'datetime', 'hco3_arterial', 'hco3_venous', 'pco2_arterial',
       'pco2_venous', 'ph_arterial', 'ph_venous', 'po2_arterial',
       'po2_venous'],
      dtype='object')

In [63]:
print(all_labs.pco2_venous.dropna().shape)
print(all_labs.pco2_arterial.dropna().shape)

(1224,)
(3241,)


In [72]:
plt.figure(figsize =(6,2))
plt.hist(all_labs.pco2_venous.dropna().values, histtype='step', density=True, color='b', bins=20)
plt.hist(all_labs.pco2_arterial.dropna().values, histtype='step', density=True, color='red', bins=20)
plt.legend(['venous', 'arterial'])
plt.title('PCO2')
plt.xlabel('mmHg')
plt.ylabel('Probability Density')
plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [73]:
plt.figure(figsize =(6,2))
plt.hist(all_labs.po2_venous.dropna().values, histtype='step', density=True, color='b', bins=20)
plt.hist(all_labs.po2_arterial.dropna().values, histtype='step', density=True, color='red', bins=20)
plt.legend(['venous', 'arterial'])
plt.title('PO2')
plt.xlabel('mmHg')
plt.ylabel('Probability Density')
plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [75]:
plt.figure(figsize =(6,2))
plt.hist(all_labs.ph_venous.dropna().values, histtype='step', density=True, color='b', bins=20)
plt.hist(all_labs.ph_arterial.dropna().values, histtype='step', density=True, color='red', bins=20)
plt.legend(['venous', 'arterial'])
plt.title('PH')
plt.xlabel('mmHg')
plt.ylabel('Probability Density')
plt.tight_layout()

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:1: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  """Entry point for launching an IPython kernel.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

### add SOFA

In [215]:
# for file in files: # ['icusleep_004.h5']:
file = files[10]

data_dir = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_v1' # TMP TODO
files = os.listdir(data_dir)


for file in files[::20]:
    study_id = file[-6:-3]

    mrn = LIST.loc[LIST.study_id==study_id, 'mrn'].values[0]
    input_file_path = os.path.join(data_dir, file)
    output_file_path = os.path.join(output_dir, file)

    # if os.path.exists(output_file_path):
    #     continue
    input_file_path

    print(file)
    data, hdr, fs  = read_in_routine(input_file_path, check_airgo_scaling=False)

    # load sofa data:
    sofa_data = pd.read_csv(os.path.join(sofa_dir, 'SOFA_ICU'+study_id+'.csv'))
    sofa_data.columns = ['sofa_respiratory','sofa_coagulatory', 'sofa_liver', 'sofa_cvs', 'sofa_cns', 'sofa_renal', 'sofa_score', 'sofa_delta_2h', 'sofa_delta_4h', 'sofa_delta_6h', 'datetime']
    sofa_data['datetime'] = pd.to_datetime(sofa_data['datetime'], infer_datetime_format=1)
    sofa_data.set_index('datetime', inplace=True)
    
    # and add to data:
    data = add_sofa_to_data(data, sofa_data)
    
        
    import matplotlib.dates as mdates


    fig, ax = plt.subplots(1,1, figsize=(10,5))
    ax.plot(data.loc[data.sofa_score>-99, 'sofa_score'],  c='black')
    ax.plot(data.loc[data.sofa_respiratory>-99, 'sofa_respiratory']+0.02, c='blue')
    ax.plot(data.loc[data.sofa_cvs>-99, 'sofa_cvs']+0.04, c='red')
    ax.plot(data.loc[data.sofa_cns>-99, 'sofa_cns']+0.06, c='gold')
    ax.plot(data.loc[data.sofa_liver>-99, 'sofa_liver']+0.08, c='purple')
    ax.plot(data.loc[data.sofa_coagulatory>-99, 'sofa_coagulatory']+0.1, c='gray')
    ax.plot(data.loc[data.sofa_renal>-99, 'sofa_renal']+0.12, c='brown')


    plt.xticks(rotation=90)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H'))
    fig.subplots_adjust(bottom=0.3)
    plt.title(f'SOFA Score for #{study_id}')
    plt.legend(['SOFA score', 'Respiratory', 'CVS', 'CNS', 'Liver', 'Coagulatory', 'Renal'], bbox_to_anchor=(1,0.5))
    plt.show()
    plt.tight_layout()
    plt.savefig(f'SOFA_{study_id}.png')

icusleep_090.h5


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

icusleep_149.h5


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


...

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

icusleep_181.h5


/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:36: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/AirGoSleep/lib/python3.6/site-packages/ipykernel_launcher.py:4: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`).
  after removing the cwd from sys.path.


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …